Fine-Tuning (Task-Specific Pretrained Model)

Goal: Start from a pretrained model (BERT, GPT, BART, etc.) and adapt it to a specific downstream task like classification, summarization, or QA.

In [ ]:
!pip install transformers datasets accelerate


In [ ]:
# ------------------------------
# Plain GPT-2 Fine-Tuning 
# ------------------------------

In [19]:
#!pip install -q transformers datasets accelerate

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

# Load tiny dataset for demo
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:100]")  # 100 examples

# Load GPT-2 tokenizer and model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

# Tokenize dataset
def tokenize_function(examples):
    tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    tokens["labels"] = tokens["input_ids"]  # next-token prediction
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Training arguments
training_args = TrainingArguments(
    output_dir="./gpt2_finetune",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    logging_steps=10,
    save_strategy="no",
    fp16=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

# Train
trainer.train()

# Generate text
prompt = "Artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_length=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


README.md: 0.00B [00:00, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/733k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/6.36M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/657k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Step,Training Loss
10,3.185000
20,2.049600
30,2.213100
40,2.014300
50,1.296300


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Artificial intelligence is a tool that can be used to improve the performance of a computer by analyzing the behavior of the human brain.


# # Motivation for Instruction Fine-Tuning

In [ ]:

1. Bridges the Gap Between Pretraining and Tasks
Pretrained models (GPT, T5, BERT) learn general language patterns from massive text.
They are not optimized for following user instructions like “Summarize this article” or “Translate to French.”
IFT adapts the model to understand and respond to explicit instructions in natural language.
    
2. Improves Model Usability

Without IFT, you have to carefully craft prompts to get the desired output.
After IFT, the model can understand instructions more reliably, reducing prompt engineering.

3. Supports Multi-Task Learning

A single instruction-fine-tuned model can:
    Summarize text
    Answer questions
    Translate languages
    Generate code
This is more flexible than task-specific fine-tuning.

4. Better Generalization

IFT teaches the model to interpret instructions instead of memorizing task-specific patterns.
Leads to strong zero-shot or few-shot performance on new tasks without extra fine-tuning.


5. Enables Human-Like Interaction

Models can respond as humans would expect from instructions:
    “Explain this like I’m 5.”
    “Write a formal email.”
    “Generate 3 examples of X.”

Makes AI more practical for real-world applications.


In [ ]:
| Aspect       | Fine-Tuning                 | Instruction Fine-Tuning          |
| ------------ | --------------------------- | -------------------------------- |
| Input Format | Task-specific input → label | Instruction + input → output     |
| Flexibility  | Low (task-specific)         | High (multi-task, zero-shot)     |
| Usability    | Needs careful prompt design | Understands natural instructions |
| Example      | Sentiment classification    | Summarization, translation, Q\&A |


# Instruction based finetuning

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
import torch

# Tiny instruction dataset
data = [
    {"instruction": "Summarize the text", "input": "AI is transforming technology.", "output": "AI is changing technology."}
]

dataset = Dataset.from_list(data)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

max_length = 128

def preprocess(example):
    # Combine instruction + input + output
    prompt = f"Instruction: {example['instruction']}\nInput: {example['input']}\nOutput: {example['output']}"
    
    encodings = tokenizer(prompt, truncation=True, padding="max_length", max_length=max_length)
    
    # Labels are the same as input_ids
    encodings["labels"] = encodings["input_ids"].copy()
    
    return encodings

tokenized_dataset = dataset.map(preprocess)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])



Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT2 has no default pad token


In [13]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ift_results",
    per_device_train_batch_size=2,   # small batch to fit t3.large
    num_train_epochs=2,
    logging_steps=1,
    save_strategy="no",
    fp16=False  # t3.large may not support FP16
)


In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)


In [16]:
trainer.train()


Step,Training Loss
1,8.417000
2,4.481300


TrainOutput(global_step=2, training_loss=6.449141502380371, metrics={'train_runtime': 6.4991, 'train_samples_per_second': 0.308, 'train_steps_per_second': 0.308, 'total_flos': 130646016000.0, 'train_loss': 6.449141502380371, 'epoch': 2.0})

In [18]:
prompt = "Instruction: Summarize the text\nInput: Artificial intelligence is changing the world.\nOutput:"

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_length=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Summarize the text
Input: Artificial intelligence is changing the world.
Output: The world is changing.
Input: The world is changing.
Output: The world is changing.
Output: The world is changing.


# Comparsion

In [ ]:
| Type                        | Input/Output Format                 | Goal                                            | Example Models       |
| --------------------------- | ----------------------------------- | ----------------------------------------------- | -------------------- |
| **Fine-Tuning**             | Standard task input → label         | Adapt pretrained model to a specific task       | BERT, GPT, BART      |
| **Instruction Fine-Tuning** | Instruction + input → output (text) | Make model follow natural language instructions | GPT, T5, FLAN, LLaMA |


## Instruction fine-tuning turns a pretrained model from a “general language engine” into a practical, instruction-following assistant, capable of handling many tasks in a human-like way

# Knowledge Distillation

Knowledge distillation is a technique where you transfer the knowledge from a large “teacher” model to a smaller “student” model.
Teacher: Usually a very large, powerful LLM (like GPT-4, LLaMA-65B).
Student: A smaller model (like GPT-2 or a compact transformer) that’s easier to deploy.

Goal: The student learns to mimic the teacher’s outputs without needing all the parameters.

In [ ]:
How Distillation Works in LLMs?

1. Teacher Generates Soft Labels

For each input, the teacher produces probabilities over the vocabulary (not just the final token).
These probabilities are richer than hard labels because they contain the teacher’s “beliefs” about alternatives.

2. Student Learns from Teacher

The student is trained to match the teacher’s output distribution (using e.g., KL divergence loss).
The student can also optionally learn from ground truth labels if available (hybrid distillation).

In [ ]:
Why distillation is powerful?

| Benefit                 | Explanation                                                                                               |
| ----------------------- | --------------------------------------------------------------------------------------------------------- |
| **Smaller model size**  | Student models have fewer parameters → easier to deploy on CPUs, mobile, or low-memory GPUs.              |
| **Faster inference**    | Smaller student → lower latency → real-time applications.                                                 |
| **Reduced cost**        | Less compute needed for training & inference.                                                             |
| **Preserves knowledge** | Student can retain much of the teacher’s generalization ability, sometimes even better for certain tasks. |


In [ ]:
Variants in LLM Distillation

Full Model Distillation: Train a small transformer to mimic the teacher on general language tasks.
Instruction Distillation: Teacher generates instruction-following responses → student learns to follow instructions.
Response Distillation: For chatbots, the teacher generates multiple responses, and the student learns to reproduce them.

In [ ]:
Differences between FT and DT

| Aspect              | Fine-Tuning                   | Distillation                                           |
| ------------------- | ----------------------------- | ------------------------------------------------------ |
| **Goal**            | Adapt to a task or domain     | Compress model size / improve efficiency               |
| **Teacher needed?** | No                            | Yes (large pretrained model)                           |
| **Data**            | Task-specific labeled dataset | Can be teacher-generated or original data              |
| **Output**          | Task-specialized model        | Smaller general-purpose or instruction-following model |


In [20]:
# Load Teacher and Student Models

from transformers import AutoTokenizer, AutoModelForCausalLM

# Teacher: larger GPT-2
teacher_model_name = "gpt2-medium"
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)
teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
teacher_model = AutoModelForCausalLM.from_pretrained(teacher_model_name)
teacher_model.eval()  # teacher is frozen

# Student: smaller GPT-2
student_model_name = "gpt2"
student_tokenizer = AutoTokenizer.from_pretrained(student_model_name)
student_tokenizer.pad_token = student_tokenizer.eos_token
student_model = AutoModelForCausalLM.from_pretrained(student_model_name)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [21]:
# Sample Distillation Dataset

from datasets import Dataset

data = [
    {"text": "Artificial intelligence is transforming technology."},
    {"text": "The stock market showed strong gains today."},
    {"text": "Climate change impacts are becoming more visible each year."},
]

dataset = Dataset.from_list(data)


In [22]:
#Tokenize Dataset

def tokenize(examples):
    return student_tokenizer(examples["text"], truncation=True, padding="max_length", max_length=32)

tokenized_dataset = dataset.map(tokenize)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [23]:
# Distillation Training Loop

import torch
from torch.nn import KLDivLoss
from torch.optim import Adam

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher_model.to(device)
student_model.to(device)

optimizer = Adam(student_model.parameters(), lr=5e-5)
loss_fn = KLDivLoss(reduction="batchmean")  # use KL divergence

for epoch in range(2):  # keeping it intenionally low for demo
    total_loss = 0
    for batch in tokenized_dataset:
        input_ids = batch["input_ids"].unsqueeze(0).to(device)
        attention_mask = batch["attention_mask"].unsqueeze(0).to(device)
        
        with torch.no_grad():
            teacher_logits = teacher_model(input_ids, attention_mask=attention_mask).logits
            teacher_probs = torch.softmax(teacher_logits / 2.0, dim=-1)  # temperature=2
        
        student_logits = student_model(input_ids, attention_mask=attention_mask).logits
        student_log_probs = torch.log_softmax(student_logits / 2.0, dim=-1)
        
        loss = loss_fn(student_log_probs, teacher_probs)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 6.6680
Epoch 2, Loss: 5.5953


In [24]:
# Test Student Model

prompt = "Artificial intelligence is"
inputs = student_tokenizer(prompt, return_tensors="pt").to(device)
outputs = student_model.generate(**inputs, max_length=30)
print(student_tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Artificial intelligence is a new field of research that has been in the works for some time now. It is a field that has been around for a


# Teacher is frozen; student learns by mimicking teacher outputs.